In [ ]:
# ========================================
# AUTOMATISCHER FARBMODUS-KOMBINATIONS-TEST
# ========================================
# Testet ALLE möglichen Kombinationen von Farbmodi zwischen beiden Bildern
# und findet die besten Kombinationen basierend auf Confidence-Score

import time
from itertools import product

# Alle verfügbaren Farbmodi
all_color_modes = ['GRAY', 'RED', 'GREEN', 'BLUE', 'LAB_L', 'LAB_A', 'LAB_B', 
                   'HSV_V', 'HSV_S', 'HSV_H', 'INVERTED', 'BINARY', 'CLAHE', 
                   'ADAPTIVE', 'EDGES']

print(f"🔍 Starte Farbmodus-Kombinations-Test...")
print(f"📊 Teste {len(all_color_modes)}² = {len(all_color_modes)**2} Kombinationen")
print(f"⏱️  Geschätzte Zeit: ~{len(all_color_modes)**2 * 2 / 60:.1f} Minuten\n")

results = []
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Verwende die aktuelle Config aus Zelle 2
current_config = {
    'superpoint': {
        'nms_radius': 3,
        'keypoint_threshold': 0.001,
        'max_keypoints': 2048
    },
    'superglue': {
        'weights': 'outdoor',
        'sinkhorn_iterations': 100,
        'match_threshold': 0.1,
    }
}

# Teste jede Kombination
total_tests = len(all_color_modes) ** 2
current_test = 0

for mode1 in all_color_modes:
    for mode2 in all_color_modes:
        current_test += 1
        
        try:
            start_time = time.time()
            
            # Lade Bilder mit unterschiedlichen Farbmodi
            img1_test = load_image_channel(osm_png_path, mode1)
            img2_test = load_image_channel(bplan_png_path, mode2)
            
            # Resize
            img1_test_resized = cv2.resize(img1_test, (IMAGE_SIZE, IMAGE_SIZE))
            img2_test_resized = cv2.resize(img2_test, (IMAGE_SIZE, IMAGE_SIZE))
            
            # Erstelle Matcher (nur einmal pro Test)
            matching_test = Matching(current_config).eval().to(device)
            
            # Konvertiere zu Tensoren
            inp1_test = frame2tensor(img1_test_resized, device)
            inp2_test = frame2tensor(img2_test_resized, device)
            
            # Führe Matching durch
            with torch.no_grad():
                pred_test = matching_test({'image0': inp1_test, 'image1': inp2_test})
            
            # Extrahiere Ergebnisse
            kpts0_test = pred_test['keypoints0'][0].cpu().numpy()
            kpts1_test = pred_test['keypoints1'][0].cpu().numpy()
            matches_test = pred_test['matches0'][0].cpu().numpy()
            confidence_test = pred_test['matching_scores0'][0].cpu().numpy()
            
            # Filtere valid matches
            valid_test = matches_test > -1
            num_matches = valid_test.sum()
            
            if num_matches > 0:
                match_conf_test = confidence_test[valid_test]
                avg_confidence = match_conf_test.mean()
                max_confidence = match_conf_test.max()
                high_conf_matches = (match_conf_test > 0.5).sum()
                very_high_conf_matches = (match_conf_test > 0.7).sum()
            else:
                avg_confidence = 0
                max_confidence = 0
                high_conf_matches = 0
                very_high_conf_matches = 0
            
            elapsed_time = time.time() - start_time
            
            results.append({
                'img1_mode': mode1,
                'img2_mode': mode2,
                'num_matches': num_matches,
                'avg_confidence': avg_confidence,
                'max_confidence': max_confidence,
                'high_conf_matches': high_conf_matches,  # >0.5
                'very_high_conf_matches': very_high_conf_matches,  # >0.7
                'time': elapsed_time
            })
            
            # Progress Update alle 25 Tests
            if current_test % 25 == 0 or current_test == total_tests:
                print(f"Progress: {current_test}/{total_tests} ({current_test/total_tests*100:.1f}%) - "
                      f"Letzter Test: {mode1}↔{mode2}: {num_matches} matches, ⌀conf: {avg_confidence:.3f}")
            
            # Cleanup
            del matching_test, inp1_test, inp2_test, pred_test
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
            
        except Exception as e:
            print(f"❌ Fehler bei {mode1}↔{mode2}: {e}")
            continue

print("\n" + "="*80)
print("🏆 TOP 5 BESTE KOMBINATIONEN")
print("="*80)

# Sortiere nach verschiedenen Kriterien
print("\n🥇 TOP 5 nach DURCHSCHNITTLICHER CONFIDENCE:")
results_sorted = sorted(results, key=lambda x: x['avg_confidence'], reverse=True)
for i, res in enumerate(results_sorted[:5], 1):
    print(f"\n{i}. ⌀ Confidence: {res['avg_confidence']:.4f}")
    print(f"   Bild 1: {res['img1_mode']:12s} ↔ Bild 2: {res['img2_mode']:12s}")
    print(f"   Matches: {res['num_matches']:4d} | High-Conf (>0.5): {res['high_conf_matches']:4d} | "
          f"Very-High (>0.7): {res['very_high_conf_matches']:4d}")
    print(f"   Max Confidence: {res['max_confidence']:.4f} | Zeit: {res['time']:.2f}s")

print("\n🥇 TOP 5 nach HIGH-CONFIDENCE MATCHES (>0.5):")
results_sorted = sorted(results, key=lambda x: x['high_conf_matches'], reverse=True)
for i, res in enumerate(results_sorted[:5], 1):
    print(f"\n{i}. High-Conf Matches: {res['high_conf_matches']}")
    print(f"   Bild 1: {res['img1_mode']:12s} ↔ Bild 2: {res['img2_mode']:12s}")
    print(f"   Total Matches: {res['num_matches']:4d} | ⌀ Conf: {res['avg_confidence']:.4f} | "
          f"Max: {res['max_confidence']:.4f}")
    print(f"   Zeit: {res['time']:.2f}s")

print("\n🥇 TOP 5 nach VERY-HIGH-CONFIDENCE MATCHES (>0.7):")
results_sorted = sorted(results, key=lambda x: x['very_high_conf_matches'], reverse=True)
for i, res in enumerate(results_sorted[:5], 1):
    print(f"\n{i}. Very-High-Conf Matches: {res['very_high_conf_matches']}")
    print(f"   Bild 1: {res['img1_mode']:12s} ↔ Bild 2: {res['img2_mode']:12s}")
    print(f"   Total Matches: {res['num_matches']:4d} | High-Conf: {res['high_conf_matches']:4d} | "
          f"⌀ Conf: {res['avg_confidence']:.4f}")
    print(f"   Zeit: {res['time']:.2f}s")

print("\n🥇 TOP 5 nach GESAMTZAHL MATCHES:")
results_sorted = sorted(results, key=lambda x: x['num_matches'], reverse=True)
for i, res in enumerate(results_sorted[:5], 1):
    print(f"\n{i}. Total Matches: {res['num_matches']}")
    print(f"   Bild 1: {res['img1_mode']:12s} ↔ Bild 2: {res['img2_mode']:12s}")
    print(f"   High-Conf: {res['high_conf_matches']:4d} | Very-High: {res['very_high_conf_matches']:4d} | "
          f"⌀ Conf: {res['avg_confidence']:.4f}")
    print(f"   Zeit: {res['time']:.2f}s")

print("\n" + "="*80)
print("✅ Farbmodus-Kombinations-Test abgeschlossen!")
print(f"\n💡 EMPFEHLUNG: Kopiere die beste Kombination in Zelle 2:")
best = results_sorted[0]
print(f"   COLOR_MODE = '{best['img1_mode']}'  # für OSM-Bild")
print(f"   img2 = load_image_channel(bplan_png_path, '{best['img2_mode']}')")